# Przewidywanie goli w kolejnych M minutach — wersja Google Colab (LSTM)

Ten notebook jest **w pełni samodzielny** — nie wymaga klonowania całego repozytorium ani lokalnych importów `src/`. Możesz go wrzucić prosto do Google Colab i uruchomić na darmowym GPU.

### Jak uruchomić w Google Colab w 3 krokach
1. **Włącz GPU**: w menu Colaba wybierz *Środowisko wykonawcze → Zmień typ środowiska wykonawczego → T4 GPU*.
2. **Wybierz źródło danych w Kroku 1**:
   - **`"upload"`** *(najprostsze i najszybsze)*: wygeneruj lokalnie mały plik `output/lstm_dataset.npz` poleceniem:
     ```bash
     python -m src.lstm_data --output output/lstm_dataset.npz
     ```
     i prześlij sam ten plik (~kilka MB) w oknie Colaba (zamiast ~270 MB surowych CSV).
   - **`"drive"`**: wrzuć `lstm_dataset.npz` (albo pliki `combined_events.csv`, `combined_team_stats.csv`, `combined_metadata.csv` / spakowany `.zip`) na Dysk Google do folderu `DRIVE_DATA_DIR`.
   - **`"local"`**: tryb do uruchamiania notatnika lokalnie na komputerze.
3. **Uruchom wszystko**: *Środowisko wykonawcze → Uruchom wszystko*.

## Krok 0 — Środowisko Google Colab i instalacja brakujących pakietów

Colab ma już wbudowane `torch`, `numpy`, `pandas`, `scipy` i `matplotlib`. Jeśli wczytujesz surowe pliki CSV zamiast gotowego `.npz`, potrzebujesz dodatkowo `duckdb` — poniższa komórka doinstaluje go automatycznie tylko wtedy, gdy go brakuje.

In [ ]:
import importlib.util
import shutil
import subprocess
import sys
import zipfile
from dataclasses import dataclass
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if importlib.util.find_spec("duckdb") is None:
    print("Instaluję brakujący pakiet duckdb...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.optimize import minimize
from scipy.special import expit
from scipy.stats import rankdata
from torch.utils.data import DataLoader, TensorDataset

print(f"Środowisko : {'Google Colab' if IN_COLAB else 'Lokalne'}")
print(f"PyTorch    : {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU        : {torch.cuda.get_device_name(0)}")

## Krok 1 — Konfiguracja (formularz Google Colab)

W prawym panelu w Colabie możesz klikać i zmieniać ustawienia bez edycji kodu. Jeśli uruchomisz ten plik poza Colabem, automatycznie użyje lokalnych danych z katalogu `output/`.

In [ ]:
#@title Ustawienia źródła danych i modelu LSTM
DATA_SOURCE = "upload"  #@param ["upload", "drive", "local"]
DRIVE_DATA_DIR = "/content/drive/MyDrive/football_lstm"  #@param {type:"string"}
HORIZON_MINUTES = 10  #@param {type:"integer"}
SEQUENCE_LENGTH = 10  #@param {type:"integer"}
BATCH_SIZE = 128  #@param {type:"integer"}
HIDDEN_SIZE = 16  #@param {type:"integer"}
NUM_LAYERS = 1  #@param {type:"integer"}
DROPOUT = 0.4  #@param {type:"number"}
LEARNING_RATE = 0.0002  #@param {type:"number"}
WEIGHT_DECAY = 0.01  #@param {type:"number"}
EPOCHS = 30  #@param {type:"integer"}
PATIENCE = 7  #@param {type:"integer"}
POS_WEIGHT_SCALE = 1.0  #@param {type:"number"}
TRAIN_STRIDE = 2  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}
N_REPLAY_MATCHES = 5  #@param {type:"integer"}
ALERT_TOP_PERCENT = 20  #@param {type:"integer"}
DOWNLOAD_MODEL = False  #@param {type:"boolean"}

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EFFECTIVE_SOURCE = DATA_SOURCE
if not IN_COLAB and DATA_SOURCE in ("upload", "drive"):
    print("Poza Google Colab — przełączam automatycznie na dane lokalne z katalogu output/.")
    EFFECTIVE_SOURCE = "local"

print(f"Źródło danych: {EFFECTIVE_SOURCE} | Urządzenie treningu: {DEVICE}")

## Krok 2 — Wbudowane funkcje przygotowania danych (bez zależności od `src/`)

Poniższa komórka zawiera kompletny kod wczytywania zarówno gotowego pliku `.npz` (`load_dataset_bundle_npz`), jak i surowych tabel CSV przez DuckDB (`prepare_dataset_from_csv_dir`). Dzięki temu ten notebook działa w 100% samodzielnie w Google Colab.

In [ ]:
FEATURE_NAMES = (
    # 1. Aktywność z minuty t (10 cech)
    "box_touches",
    "opp_box_touches",
    "box_entries",
    "final_third_touches",
    "passes_final_third_acc",
    "shots_in_box",
    "xg_proxy",
    "key_passes",
    "high_recoveries",
    "opp_own_third_turnovers",
    # 2. Narastająca przewaga od początku meczu do minuty t (6 cech)
    "box_touches_rate",
    "box_entries_diff_rate",
    "poss_share_cum",
    "field_tilt_cum",
    "avg_x_diff_cum",
    "clearances_diff_rate",
    # 3. Stan meczu i czas (4 cechy)
    "score_diff",
    "man_advantage",
    "minute_norm",
    "is_home",
)

FEATURE_DESCRIPTIONS = {
    "box_touches": "Własne kontakty w polu karnym rywala w minucie t (x>=83, 21<=y<=79).",
    "opp_box_touches": "Kontakty rywala w naszym polu karnym w minucie t.",
    "box_entries": "Zdarzenia kończące się w polu karnym rywala w minucie t (end_x>=83).",
    "final_third_touches": "Kontakty w tercji ataku w minucie t (x>=66).",
    "passes_final_third_acc": "Celne podania kończące się w tercji ataku w minucie t.",
    "shots_in_box": "Strzały oddane z pola karnego w minucie t.",
    "xg_proxy": "Geometryczna suma jakości strzałów w minucie t (dystans, big chance, główka, karny).",
    "key_passes": "Podania kluczowe (otwierające strzał) w minucie t.",
    "high_recoveries": "Odbiory piłki na połowie rywala (x>=60) w minucie t.",
    "opp_own_third_turnovers": "Straty rywala w jego własnej tercji obronnej (x<=35).",
    "box_touches_rate": "Skumulowane własne kontakty w polu karnym na minutę gry do t.",
    "box_entries_diff_rate": "Skumulowana różnica wejść w pole karne (my - rywal) na minutę.",
    "poss_share_cum": "Skumulowany udział posiadania z team_stats: my / (my + rywal).",
    "field_tilt_cum": "Skumulowany udział celnych podań w tercji ataku: my / (my + rywal).",
    "avg_x_diff_cum": "Różnica średniej wysokości gry od początku meczu (my - rywal).",
    "clearances_diff_rate": "Skumulowana różnica wybić na minutę (rywal - my; więcej = oblężenie bramki rywala).",
    "score_diff": "Aktualny bilans bramek w minucie t (samobóje zaliczone drużynie zyskującej).",
    "man_advantage": "Przewaga liczebna po czerwonych kartkach w minucie t (rywal - my).",
    "minute_norm": "Rozszerzona minuta meczu podzielona przez 90.0.",
    "is_home": "1.0 dla gospodarzy, 0.0 dla gości.",
}

META_COLUMNS = (
    "match_id", "team_id", "team_name", "opp_team_name",
    "is_home", "period_value", "minute", "score_diff",
)
_STRING_META_COLS = {"team_name", "opp_team_name"}


@dataclass
class SequenceSplit:
    x: np.ndarray
    y: np.ndarray
    meta: pd.DataFrame


@dataclass
class LSTMDatasetBundle:
    train: SequenceSplit
    val: SequenceSplit
    test: SequenceSplit
    feature_names: tuple
    feature_mean: np.ndarray
    feature_std: np.ndarray
    sequence_length: int
    horizon_minutes: int


def load_dataset_bundle_npz(path: Path | str) -> LSTMDatasetBundle:
    """Wczytuje spakowany plik .npz bez użycia pickle (bezpieczne między wersjami Pythona)."""
    with np.load(Path(path), allow_pickle=False) as data:
        splits = {}
        for prefix in ("train", "val", "test"):
            meta_dict = {
                col: (
                    [str(v) for v in data[f"{prefix}_meta_{col}"].tolist()]
                    if col in _STRING_META_COLS
                    else data[f"{prefix}_meta_{col}"]
                )
                for col in META_COLUMNS
            }
            splits[prefix] = SequenceSplit(
                x=data[f"{prefix}_x"].astype(np.float32),
                y=data[f"{prefix}_y"].astype(np.float32),
                meta=pd.DataFrame(meta_dict, columns=list(META_COLUMNS)),
            )
        return LSTMDatasetBundle(
            train=splits["train"],
            val=splits["val"],
            test=splits["test"],
            feature_names=tuple(str(n) for n in data["feature_names"].tolist()),
            feature_mean=data["feature_mean"].astype(np.float32),
            feature_std=data["feature_std"].astype(np.float32),
            sequence_length=int(data["sequence_length"]),
            horizon_minutes=int(data["horizon_minutes"]),
        )


_PANEL_SQL = """
WITH half_bounds AS (
    SELECT CAST(match_id AS BIGINT) AS match_id, CAST(period_value AS INT) AS period_value,
           MIN(CAST(expanded_minute AS INT)) AS min_t, MAX(CAST(expanded_minute AS INT)) AS max_t
    FROM events WHERE CAST(period_value AS INT) IN (1, 2) AND expanded_minute IS NOT NULL
    GROUP BY 1, 2
),
teams AS (
    SELECT CAST(match_id AS BIGINT) AS match_id, CAST(home_team_id AS BIGINT) AS team_id,
           CAST(away_team_id AS BIGINT) AS opp_team_id, CAST(home_team_name AS VARCHAR) AS team_name,
           CAST(away_team_name AS VARCHAR) AS opp_team_name, 1.0 AS is_home FROM metadata
    UNION ALL
    SELECT CAST(match_id AS BIGINT) AS match_id, CAST(away_team_id AS BIGINT) AS team_id,
           CAST(home_team_id AS BIGINT) AS opp_team_id, CAST(away_team_name AS VARCHAR) AS team_name,
           CAST(home_team_name AS VARCHAR) AS opp_team_name, 0.0 AS is_home FROM metadata
),
grid AS (
    SELECT hb.match_id, hb.period_value, tm.team_id, tm.opp_team_id,
           tm.team_name, tm.opp_team_name, tm.is_home, gs.t
    FROM half_bounds hb JOIN teams tm ON hb.match_id = tm.match_id
    CROSS JOIN LATERAL generate_series(hb.min_t, hb.max_t) AS gs(t)
),
ev_goals AS (
    SELECT CAST(e.match_id AS BIGINT) AS match_id, CAST(e.period_value AS INT) AS period_value,
           CASE WHEN COALESCE(CAST(e.is_own_goal AS BOOLEAN), FALSE) THEN tm.opp_team_id
                ELSE CAST(e.team_id AS BIGINT) END AS scoring_team_id,
           CAST(e.expanded_minute AS INT) AS t, COUNT(*) AS goals_scored
    FROM events e JOIN teams tm ON CAST(e.match_id AS BIGINT) = tm.match_id AND CAST(e.team_id AS BIGINT) = tm.team_id
    WHERE COALESCE(CAST(e.is_goal AS BOOLEAN), FALSE) AND CAST(e.period_value AS INT) IN (1, 2)
      AND e.expanded_minute IS NOT NULL GROUP BY 1, 2, 3, 4
),
ev_min AS (
    SELECT CAST(match_id AS BIGINT) AS match_id, CAST(period_value AS INT) AS period_value,
           CAST(team_id AS BIGINT) AS team_id, CAST(expanded_minute AS INT) AS t,
           AVG(CAST(x AS DOUBLE)) AS ev_avg_x,
           SUM(CASE WHEN COALESCE(CAST(is_touch AS BOOLEAN), FALSE) AND x >= 83.0 AND y BETWEEN 21.0 AND 79.0 THEN 1 ELSE 0 END) AS box_touches,
           SUM(CASE WHEN end_x >= 83.0 AND end_y BETWEEN 21.0 AND 79.0 THEN 1 ELSE 0 END) AS box_entries,
           SUM(CASE WHEN COALESCE(CAST(is_touch AS BOOLEAN), FALSE) AND x >= 66.0 THEN 1 ELSE 0 END) AS final_third_touches,
           SUM(CASE WHEN event_type = 'Pass' AND outcome_type = 'Successful' AND end_x >= 66.0 THEN 1 ELSE 0 END) AS passes_final_third_acc,
           SUM(CASE WHEN COALESCE(CAST(is_shot AS BOOLEAN), FALSE) AND x >= 83.0 AND y BETWEEN 21.0 AND 79.0 THEN 1 ELSE 0 END) AS shots_in_box,
           SUM(CASE WHEN COALESCE(CAST(is_shot AS BOOLEAN), FALSE) THEN CASE WHEN COALESCE(CAST(is_penalty AS BOOLEAN), FALSE) THEN 0.76 ELSE
               1.0 / (1.0 + EXP(-(-0.6 - 0.12 * SQRT(POWER((100.0 - COALESCE(CAST(x AS DOUBLE), 85.0)) * 1.05, 2)
               + POWER((50.0 - COALESCE(CAST(y AS DOUBLE), 50.0)) * 0.68, 2))
               + 1.4 * CAST(COALESCE(CAST(is_big_chance AS BOOLEAN), FALSE) AS INT)
               - 0.6 * CASE WHEN body_part = 'Head' THEN 1 ELSE 0 END))) END ELSE 0.0 END) AS xg_proxy,
           SUM(CAST(COALESCE(CAST(is_key_pass AS BOOLEAN), FALSE) AS INT)) AS key_passes,
           SUM(CASE WHEN event_type = 'BallRecovery' AND x >= 60.0 THEN 1 ELSE 0 END) AS high_recoveries,
           SUM(CASE WHEN ((event_type = 'Pass' AND outcome_type = 'Unsuccessful') OR event_type = 'Dispossessed') AND x <= 35.0 THEN 1 ELSE 0 END) AS own_third_turnovers,
           SUM(CASE WHEN COALESCE(CAST(is_red_card AS BOOLEAN), FALSE) OR COALESCE(CAST(is_second_yellow_card AS BOOLEAN), FALSE) THEN 1 ELSE 0 END) AS red_cards
    FROM events WHERE CAST(period_value AS INT) IN (1, 2) AND expanded_minute IS NOT NULL GROUP BY 1, 2, 3, 4
),
ts_min AS (
    SELECT CAST(match_id AS BIGINT) AS match_id, CAST(team_id AS BIGINT) AS team_id, CAST(minute AS INT) AS t,
           COALESCE(MAX(CASE WHEN stat_name = 'possession' THEN CAST(value AS DOUBLE) END), 0.0) AS poss_cnt,
           COALESCE(MAX(CASE WHEN stat_name = 'clearances' THEN CAST(value AS DOUBLE) END), 0.0) AS clearances
    FROM team_stats WHERE stat_name IN ('possession', 'clearances') AND minute IS NOT NULL GROUP BY 1, 2, 3
),
team_cum AS (
    SELECT g.match_id, g.period_value, g.team_id, g.opp_team_id, g.team_name, g.opp_team_name, g.is_home, g.t,
           CAST(COALESCE(eg.goals_scored, 0) AS INT) AS goals_scored,
           CAST(COALESCE(ev.box_touches, 0) AS DOUBLE) AS box_touches,
           CAST(COALESCE(ev.box_entries, 0) AS DOUBLE) AS box_entries,
           CAST(COALESCE(ev.final_third_touches, 0) AS DOUBLE) AS final_third_touches,
           CAST(COALESCE(ev.passes_final_third_acc, 0) AS DOUBLE) AS passes_final_third_acc,
           CAST(COALESCE(ev.shots_in_box, 0) AS DOUBLE) AS shots_in_box,
           CAST(COALESCE(ev.xg_proxy, 0.0) AS DOUBLE) AS xg_proxy,
           CAST(COALESCE(ev.key_passes, 0) AS DOUBLE) AS key_passes,
           CAST(COALESCE(ev.high_recoveries, 0) AS DOUBLE) AS high_recoveries,
           CAST(COALESCE(ev.own_third_turnovers, 0) AS DOUBLE) AS own_third_turnovers,
           SUM(COALESCE(ev.box_touches, 0)) OVER wm AS box_touches_cum,
           SUM(COALESCE(ev.box_entries, 0)) OVER wm AS box_entries_cum,
           SUM(COALESCE(ev.passes_final_third_acc, 0)) OVER wm AS passes_f3_acc_cum,
           SUM(COALESCE(ts.poss_cnt, 0.0)) OVER wm AS poss_cum,
           SUM(COALESCE(ts.clearances, 0.0)) OVER wm AS clearances_cum,
           COALESCE(AVG(ev.ev_avg_x) OVER wm, 50.0) AS avg_x_cum,
           SUM(COALESCE(eg.goals_scored, 0)) OVER wm AS goals_for_cum,
           SUM(COALESCE(ev.red_cards, 0)) OVER wm AS red_cards_cum
    FROM grid g
    LEFT JOIN ev_goals eg ON g.match_id=eg.match_id AND g.period_value=eg.period_value AND g.team_id=eg.scoring_team_id AND g.t=eg.t
    LEFT JOIN ev_min ev ON g.match_id=ev.match_id AND g.period_value=ev.period_value AND g.team_id=ev.team_id AND g.t=ev.t
    LEFT JOIN ts_min ts ON g.match_id=ts.match_id AND g.team_id=ts.team_id AND g.t=ts.t
    WINDOW wm AS (PARTITION BY g.match_id, g.team_id ORDER BY g.period_value, g.t ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
)
SELECT me.match_id, me.period_value, me.team_id, me.team_name, me.opp_team_name, me.t, me.goals_scored,
       me.box_touches, opp.box_touches AS opp_box_touches, me.box_entries, me.final_third_touches,
       me.passes_final_third_acc, me.shots_in_box, me.xg_proxy, me.key_passes, me.high_recoveries,
       opp.own_third_turnovers AS opp_own_third_turnovers,
       me.box_touches_cum / GREATEST(me.t + 1.0, 1.0) AS box_touches_rate,
       (me.box_entries_cum - opp.box_entries_cum) / GREATEST(me.t + 1.0, 1.0) AS box_entries_diff_rate,
       CASE WHEN (me.poss_cum + opp.poss_cum) > 0.0 THEN me.poss_cum / (me.poss_cum + opp.poss_cum) ELSE 0.5 END AS poss_share_cum,
       CASE WHEN (me.passes_f3_acc_cum + opp.passes_f3_acc_cum) > 0.0 THEN me.passes_f3_acc_cum / (me.passes_f3_acc_cum + opp.passes_f3_acc_cum) ELSE 0.5 END AS field_tilt_cum,
       me.avg_x_cum - opp.avg_x_cum AS avg_x_diff_cum,
       (opp.clearances_cum - me.clearances_cum) / GREATEST(me.t + 1.0, 1.0) AS clearances_diff_rate,
       CAST(me.goals_for_cum - opp.goals_for_cum AS DOUBLE) AS score_diff,
       CAST(opp.red_cards_cum - me.red_cards_cum AS DOUBLE) AS man_advantage,
       CAST(me.t AS DOUBLE) / 90.0 AS minute_norm, me.is_home
FROM team_cum me
JOIN team_cum opp ON me.match_id=opp.match_id AND me.period_value=opp.period_value AND me.t=opp.t AND me.opp_team_id=opp.team_id
ORDER BY me.match_id, me.team_id, me.period_value, me.t
"""


def prepare_dataset_from_csv_dir(
    csv_dir: Path,
    sequence_length: int,
    horizon_minutes: int,
    seed: int = 42,
) -> LSTMDatasetBundle:
    """Buduje LSTMDatasetBundle z plików combined_*.csv lub match_*_*.csv w podanym folderze."""
    con = duckdb.connect()
    try:
        has_per_match = (
            any(csv_dir.glob("match_*_events.csv"))
            and any(csv_dir.glob("match_*_team_stats.csv"))
            and any(csv_dir.glob("match_*_metadata.csv"))
        )
        if has_per_match:
            con.execute("CREATE TABLE events AS SELECT match_id, period_value, team_id, expanded_minute, event_type, outcome_type, is_touch, is_shot, is_goal, is_own_goal, is_big_chance, is_key_pass, is_penalty, is_red_card, is_second_yellow_card, body_part, x, y, end_x, end_y FROM read_csv_auto(?, union_by_name=True)", [str(csv_dir / "match_*_events.csv")])
            con.execute("CREATE TABLE team_stats AS SELECT match_id, team_id, stat_name, minute, value FROM read_csv_auto(?, union_by_name=True)", [str(csv_dir / "match_*_team_stats.csv")])
            con.execute("CREATE TABLE metadata AS SELECT match_id, home_team_id, away_team_id, home_team_name, away_team_name FROM read_csv_auto(?, union_by_name=True)", [str(csv_dir / "match_*_metadata.csv")])
        else:
            search_dir = csv_dir / "combined" if (csv_dir / "combined" / "combined_events.csv").exists() else csv_dir
            for tbl in ("events", "team_stats", "metadata"):
                p = search_dir / f"combined_{tbl}.csv"
                if not p.exists():
                    raise FileNotFoundError(f"Brak pliku {p} ani plików match_*_{tbl}.csv w {csv_dir}")
                con.execute(f"CREATE TABLE {tbl} AS SELECT * FROM read_csv_auto(?, header=True)", [str(p)])
        panel = con.execute(_PANEL_SQL).fetchdf()
    finally:
        con.close()

    panel = panel.sort_values(["match_id", "team_id", "period_value", "t"]).reset_index(drop=True)
    goals = panel["goals_scored"].to_numpy(dtype=np.int32)
    mins = panel["t"].to_numpy(dtype=np.int32)
    feat_mat = panel[list(FEATURE_NAMES)].to_numpy(dtype=np.float32)

    seq_x, seq_y, meta_idx = [], [], []
    span = sequence_length - 1
    for idx in panel.groupby(["match_id", "team_id", "period_value"], sort=False).groups.values():
        pos = np.asarray(list(idx), dtype=np.int64)
        half_t, half_g = mins[pos], goals[pos]
        half_max_t = int(half_t[-1])
        for i in range(span, len(pos)):
            if int(half_t[i]) + horizon_minutes > half_max_t:
                continue
            if int(half_t[i]) - int(half_t[i - span]) != span:
                continue
            seq_x.append(feat_mat[pos[i - span : i + 1]])
            seq_y.append(1.0 if np.any(half_g[i + 1 : i + 1 + horizon_minutes] > 0) else 0.0)
            meta_idx.append(int(pos[i]))

    x_all = np.stack(seq_x, axis=0).astype(np.float32)
    y_all = np.asarray(seq_y, dtype=np.float32)
    meta_all = panel.iloc[meta_idx].rename(columns={"t": "minute"})[list(META_COLUMNS)].reset_index(drop=True)

    unique_m = np.sort(meta_all["match_id"].unique())
    np.random.default_rng(seed).shuffle(unique_m)
    n_test = max(1, round(len(unique_m) * 0.15))
    n_val = max(1, round(len(unique_m) * 0.15))
    test_ids = set(unique_m[:n_test].tolist())
    val_ids = set(unique_m[n_test : n_test + n_val].tolist())
    train_ids = set(unique_m[n_test + n_val :].tolist())

    m_col = meta_all["match_id"].to_numpy(dtype=np.int64)
    tr_mask = np.isin(m_col, list(train_ids))
    mean = np.nanmean(x_all[tr_mask], axis=(0, 1)).astype(np.float32)
    std = np.nanstd(x_all[tr_mask], axis=(0, 1))
    std = np.where((std > 1e-8) & np.isfinite(std), std, 1.0).astype(np.float32)

    def _make_split(mask: np.ndarray) -> SequenceSplit:
        scaled = np.clip(np.nan_to_num((x_all[mask] - mean) / std, nan=0.0), -5.0, 5.0).astype(np.float32)
        return SequenceSplit(x=scaled, y=y_all[mask], meta=meta_all.loc[mask].reset_index(drop=True))

    return LSTMDatasetBundle(
        train=_make_split(tr_mask),
        val=_make_split(np.isin(m_col, list(val_ids))),
        test=_make_split(np.isin(m_col, list(test_ids))),
        feature_names=FEATURE_NAMES,
        feature_mean=mean,
        feature_std=std,
        sequence_length=sequence_length,
        horizon_minutes=horizon_minutes,
    )

## Krok 3 — Wczytanie danych w Google Colab (lub lokalnie) i podgląd cech

Komórka automatycznie wybiera właściwą ścieżkę:
- Jeśli w folderze znajduje się plik `.npz` (np. `lstm_dataset.npz`), ładuje się błyskawicznie w ~0,1 s.
- Jeśli podasz surowe pliki `.csv` (lub archiwum `.zip` z plikami CSV), automatycznie je rozpakuje i przetworzy w DuckDB.

In [ ]:
def load_from_directory(data_dir: Path) -> LSTMDatasetBundle:
    """Ładuje lstm_dataset.npz, dowolny plik *.npz, lub buduje dane z CSV w data_dir."""
    npz_candidates = [data_dir / "lstm_dataset.npz", *sorted(data_dir.glob("*.npz"))]
    for npz_path in npz_candidates:
        if npz_path.exists():
            print(f"Wczytuję gotowy zbiór .npz: {npz_path}")
            return load_dataset_bundle_npz(npz_path)
    print(f"Buduję sekwencje z plików CSV w: {data_dir}")
    return prepare_dataset_from_csv_dir(
        csv_dir=data_dir,
        sequence_length=SEQUENCE_LENGTH,
        horizon_minutes=HORIZON_MINUTES,
        seed=SEED,
    )


DRIVE_MOUNTED = False
if EFFECTIVE_SOURCE == "upload":
    from google.colab import files

    upload_dir = Path("/content/colab_data")
    upload_dir.mkdir(parents=True, exist_ok=True)
    print("Wybierz z dysku plik lstm_dataset.npz (albo pliki CSV / .zip)...")
    uploaded = files.upload()
    for fname, content in uploaded.items():
        dest = upload_dir / Path(fname).name
        dest.write_bytes(content)
        if dest.suffix.lower() == ".zip":
            with zipfile.ZipFile(dest, "r") as zf:
                zf.extractall(upload_dir)
    bundle = load_from_directory(upload_dir)
    MODEL_PATH = Path("/content/lstm_goal_predictor.pt")
elif EFFECTIVE_SOURCE == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
    drive_path = Path(DRIVE_DATA_DIR)
    bundle = load_from_directory(drive_path)
    MODEL_PATH = drive_path / "lstm_goal_predictor.pt"
else:
    project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    local_dir = project_root / "output"
    bundle = load_from_directory(local_dir)
    MODEL_PATH = local_dir / "lstm_goal_predictor.pt"

print(f"\nHoryzont M = {bundle.horizon_minutes} min | Długość sekwencji L = {bundle.sequence_length} min")
for split_name, split in [("train", bundle.train), ("val", bundle.val), ("test", bundle.test)]:
    print(
        f"  {split_name:<5} | mecze: {split.meta['match_id'].nunique():>3} | "
        f"próbki: {len(split.y):>6,} | kształt: {split.x.shape} | "
        f"odsetek goli: {100.0 * float(split.y.mean()):.2f}%"
    )

glossary = pd.DataFrame(
    [(i + 1, name, FEATURE_DESCRIPTIONS[name]) for i, name in enumerate(bundle.feature_names)],
    columns=["#", "Cecha", "Znaczenie"],
)
display(glossary)

# Podgląd jednej odtworzonej (nieskalowanej) sekwencji treningowej.
sample_meta = bundle.train.meta.iloc[0]
sample_raw = bundle.train.x[0] * bundle.feature_std + bundle.feature_mean
end_t = int(sample_meta["minute"])
print(
    f"Przykładowa sekwencja: {sample_meta['team_name']} vs {sample_meta['opp_team_name']} "
    f"(mecz {sample_meta['match_id']}, połowa {sample_meta['period_value']}, "
    f"koniec w minucie {end_t}) -> cel = {int(bundle.train.y[0])}"
)
display(pd.DataFrame(sample_raw, index=range(end_t - bundle.sequence_length + 1, end_t + 1), columns=bundle.feature_names).round(3))

## Krok 4 — Przerzedzenie okien treningowych i PyTorch DataLoaders

Sąsiednie okna w zbiorze `train` różnią się tylko o 1 minutę, więc przy małej liczbie meczów sieć szybko uczy się ich na pamięć. Przerzedzenie co `TRAIN_STRIDE` minut (`2` = co druga minuta) **wyłącznie na treningu** zmniejsza korelację próbek i ogranicza przeuczenie. Zbiory `val` i `test` zostają w 100% pełne.

`TensorDataset` łączy tensory `(x, y)`, a `DataLoader` podaje je w paczkach po `BATCH_SIZE` próbek.

In [ ]:
def subsample_train_split(split: SequenceSplit, stride: int) -> SequenceSplit:
    """Bierze co stride-tą minutę w każdej połowie meczu (tylko na potrzeby treningu)."""
    if stride <= 1:
        return split
    keep_indices = []
    grouped = split.meta.groupby(["match_id", "team_id", "period_value"], sort=False).groups
    for idx in grouped.values():
        pos = sorted(int(i) for i in idx)
        keep_indices.extend(pos[::stride])
    keep = np.asarray(sorted(keep_indices), dtype=np.int64)
    return SequenceSplit(
        x=split.x[keep],
        y=split.y[keep],
        meta=split.meta.iloc[keep].reset_index(drop=True),
    )


def make_loader(split: SequenceSplit, batch_size: int, shuffle: bool) -> DataLoader:
    """Konwertuje SequenceSplit na PyTorch DataLoader."""
    dataset = TensorDataset(
        torch.tensor(split.x, dtype=torch.float32),
        torch.tensor(split.y, dtype=torch.float32),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


train_split = subsample_train_split(bundle.train, TRAIN_STRIDE)
print(
    f"Trening (stride={TRAIN_STRIDE}): {len(bundle.train.y):,} -> {len(train_split.y):,} okien "
    f"(odsetek goli: {100.0 * float(train_split.y.mean()):.2f}%)"
)

train_loader = make_loader(train_split, BATCH_SIZE, shuffle=True)
val_loader = make_loader(bundle.val, BATCH_SIZE, shuffle=False)
test_loader = make_loader(bundle.test, BATCH_SIZE, shuffle=False)

bx, by = next(iter(train_loader))
print("Kształt paczki x:", tuple(bx.shape), "-> (batch_size, seq_len, num_features)")
print("Kształt paczki y:", tuple(by.shape), "-> (batch_size,)")

## Krok 5 — Definicja modelu LSTM

Sieć `GoalPredictorLSTM` krok po kroku:
1. `nn.LSTM` przetwarza po kolei minuty `t - 9 .. t`, budując wektor pamięci o rozmiarze `HIDDEN_SIZE`.
2. Pobieramy ostatni stan ukryty `h_n[-1]` z minuty `t`.
3. `nn.Dropout` ogranicza przeuczenie.
4. `nn.Linear(HIDDEN_SIZE, 1)` zwraca surowy wynik (**logit**), który po przejściu przez kalibrację i `sigmoid` daje prawdopodobieństwo gola.

In [ ]:
class GoalPredictorLSTM(nn.Module):
    """Prosty, czytelny klasyfikator LSTM do przewidywania gola w kolejnych M minutach."""

    def __init__(
        self,
        num_features: int,
        hidden_size: int = 32,
        num_layers: int = 1,
        dropout: float = 0.25,
    ) -> None:
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=num_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x:   (batch_size, seq_len, num_features)
        # h_n: (num_layers, batch_size, hidden_size)
        _, (h_n, _) = self.lstm(x)
        last_hidden = h_n[-1]                         # (batch_size, hidden_size)
        logits = self.head(self.dropout(last_hidden)) # (batch_size, 1)
        return logits.squeeze(-1)                     # (batch_size,)


model = GoalPredictorLSTM(
    num_features=len(bundle.feature_names),
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

print(model)
print(f"\nLiczba trenowalnych parametrów: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Krok 6 — Funkcja straty (`POS_WEIGHT_SCALE`) i metryki oceny

- Zbyt silne `pos_weight` (~6×) wypycha skrajne logity i psuje 10. decyl kalibracji, dlatego domyślnie używamy łagodnego `POS_WEIGHT_SCALE = 1.0` (standardowe `BCEWithLogitsLoss`; możesz przetestować np. `1.5`), a prawdopodobieństwa dokładnie skaluje potem kalibracja Platta.
- Oceniamy model zestawem metryk dla niezbalansowanych klas: **ROC-AUC**, **PR-AUC**, **Log-Loss**, **Brier score**, **Lift w Top-K%** oraz **Precision / Recall / F1 / FPR** dla progu alarmowego.

In [ ]:
pos_count = float((train_split.y == 1).sum())
neg_count = float((train_split.y == 0).sum())
pos_weight_value = float(POS_WEIGHT_SCALE)

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_value], device=DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
print(f"Trening: pozytywne={int(pos_count):,}, negatywne={int(neg_count):,}, pos_weight={pos_weight_value:.2f}")


def roc_auc_score_np(y_true: np.ndarray, y_score: np.ndarray) -> float:
    n_pos = int((y_true == 1).sum())
    n_neg = int((y_true == 0).sum())
    if n_pos == 0 or n_neg == 0:
        return 0.5
    ranks = rankdata(y_score, method="average")
    return float((ranks[y_true == 1].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


def pr_auc_score_np(y_true: np.ndarray, y_score: np.ndarray) -> float:
    n_pos = int((y_true == 1).sum())
    if n_pos == 0:
        return 0.0
    order = np.argsort(-y_score)
    y_sorted = y_true[order]
    precision = np.cumsum(y_sorted) / np.arange(1, len(y_sorted) + 1)
    return float(np.sum(precision * y_sorted) / n_pos)


def brier_score_np(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    return float(np.mean((y_prob - y_true) ** 2))


def log_loss_np(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    p = np.clip(y_prob, 1e-7, 1.0 - 1e-7)
    return float(-np.mean(y_true * np.log(p) + (1.0 - y_true) * np.log(1.0 - p)))


def top_k_lift(y_true: np.ndarray, y_prob: np.ndarray, fraction: float = 0.10) -> tuple[float, float]:
    k = max(1, int(len(y_true) * fraction))
    top_rate = float(y_true[np.argsort(-y_prob)[:k]].mean())
    return top_rate, top_rate / max(float(y_true.mean()), 1e-8)


def classification_metrics_at_threshold(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    threshold: float,
) -> dict[str, float]:
    """Liczy TP/FP/FN/TN oraz Precision, Recall, F1, Specificity i FPR dla danego progu."""
    y_pred = y_prob >= threshold
    y_pos = y_true >= 0.5
    tp = int(np.sum(y_pred & y_pos))
    fp = int(np.sum(y_pred & ~y_pos))
    fn = int(np.sum(~y_pred & y_pos))
    tn = int(np.sum(~y_pred & ~y_pos))

    precision = tp / max(tp + fp, 1) if (tp + fp) > 0 else 0.0
    recall = tp / max(tp + fn, 1) if (tp + fn) > 0 else 0.0
    f1 = (2.0 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    specificity = tn / max(tn + fp, 1) if (tn + fp) > 0 else 0.0
    fpr = fp / max(fp + tn, 1) if (fp + tn) > 0 else 0.0
    alert_rate = (tp + fp) / max(len(y_pos), 1)
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "specificity": float(specificity),
        "fpr": float(fpr),
        "alert_rate": float(alert_rate),
    }

## Krok 7 — Trening z Early Stopping i wykresy uczenia

Po każdej epoce sprawdzamy walidacyjne **ROC-AUC** i zachowujemy najlepsze wagi. Gdy wynik nie rośnie przez `PATIENCE` epok, zatrzymujemy trening.

In [ ]:
def train_one_epoch(net: nn.Module, loader: DataLoader) -> float:
    net.train()
    total_loss, total_rows = 0.0, 0
    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(net(x_batch), y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += float(loss.item()) * len(y_batch)
        total_rows += len(y_batch)
    return total_loss / total_rows


@torch.no_grad()
def predict_logits(net: nn.Module, loader: DataLoader) -> tuple[float, np.ndarray, np.ndarray]:
    net.eval()
    total_loss, total_rows = 0.0, 0
    logits_list, targets_list = [], []
    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
        logits = net(x_batch)
        loss = criterion(logits, y_batch)
        total_loss += float(loss.item()) * len(y_batch)
        total_rows += len(y_batch)
        logits_list.append(logits.cpu().numpy())
        targets_list.append(y_batch.cpu().numpy())
    return total_loss / total_rows, np.concatenate(logits_list), np.concatenate(targets_list)


best_val_auc, best_state, no_improve, history = -1.0, None, 0, []
for epoch in range(1, EPOCHS + 1):
    tr_loss = train_one_epoch(model, train_loader)
    val_loss, val_logits, val_y = predict_logits(model, val_loader)
    val_auc = roc_auc_score_np(val_y, val_logits)
    val_pr = pr_auc_score_np(val_y, val_logits)
    history.append({"epoch": epoch, "train_loss": tr_loss, "val_loss": val_loss, "val_auc": val_auc})

    marker = ""
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
        marker = " *best*"
    else:
        no_improve += 1

    print(f"Epoka {epoch:2d}/{EPOCHS} | train_loss={tr_loss:.4f} | val_loss={val_loss:.4f} | val_ROC-AUC={val_auc:.4f} | val_PR-AUC={val_pr:.4f}{marker}")
    if no_improve >= PATIENCE:
        print(f"Early stopping po {epoch} epokach (najlepsze val ROC-AUC: {best_val_auc:.4f}).")
        break

model.load_state_dict(best_state)
hist_df = pd.DataFrame(history)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(hist_df["epoch"], hist_df["train_loss"], marker="o", label="Train loss")
ax1.plot(hist_df["epoch"], hist_df["val_loss"], marker="o", label="Val loss")
ax1.set_title("Ważona strata BCE")
ax1.set_xlabel("Epoka")
ax1.legend()
ax2.plot(hist_df["epoch"], hist_df["val_auc"], marker="o", color="tab:green", label="Val ROC-AUC")
ax2.set_title("Walidacyjne ROC-AUC")
ax2.set_xlabel("Epoka")
ax2.legend()
plt.tight_layout()
plt.show()

## Krok 8 — Kalibracja Platta i pełna tabela metryk na meczach testowych

Kalibracja Platta dopasowuje na zbiorze **walidacyjnym** dwa współczynniki `(a, b)`, aby `p = sigmoid(a * logit + b)` zwracał dobrze wyskalowane prawdopodobieństwa na niewidzianych meczach **testowych**. Próg alarmowy (`ALERT_TOP_PERCENT`) również wyznaczamy wyłącznie na zbiorze **walidacyjnym**.

Poniżej zestawiamy **wszystkie kluczowe metryki** na historycznym zbiorze testowym przeciwko stałemu baseline'owi oraz rysujemy rozkład prawdopodobieństw i krzywą kalibracji.

In [ ]:
def fit_platt_scaling(logits: np.ndarray, y_true: np.ndarray) -> tuple[float, float]:
    def nll(params: np.ndarray) -> float:
        p = expit(params[0] * logits + params[1])
        return float(-np.mean(y_true * np.log(p + 1e-9) + (1.0 - y_true) * np.log(1.0 - p + 1e-9)))
    res = minimize(nll, x0=np.array([1.0, 0.0]), method="Nelder-Mead")
    return float(res.x[0]), float(res.x[1])


_, val_logits, val_y = predict_logits(model, val_loader)
platt_a, platt_b = fit_platt_scaling(val_logits, val_y)
val_probs = expit(platt_a * val_logits + platt_b)
alert_threshold = float(np.quantile(val_probs, 1.0 - ALERT_TOP_PERCENT / 100.0))

_, test_logits, test_y = predict_logits(model, test_loader)
test_probs = expit(platt_a * test_logits + platt_b)

base_prob = float(bundle.train.y.mean())
base_probs = np.full_like(test_y, base_prob)

test_roc = roc_auc_score_np(test_y, test_probs)
test_pr = pr_auc_score_np(test_y, test_probs)
test_logloss = log_loss_np(test_y, test_probs)
base_logloss = log_loss_np(test_y, base_probs)
test_brier = brier_score_np(test_y, test_probs)
base_brier = brier_score_np(test_y, base_probs)
brier_skill_pct = 100.0 * (1.0 - test_brier / max(base_brier, 1e-12))

top10_rate, top10_lift = top_k_lift(test_y, test_probs, fraction=0.10)
top20_rate, top20_lift = top_k_lift(test_y, test_probs, fraction=0.20)
mean_p_pos = float(test_probs[test_y == 1].mean())
mean_p_neg = float(test_probs[test_y == 0].mean())
lstm_cls = classification_metrics_at_threshold(test_y, test_probs, alert_threshold)
base_cls = classification_metrics_at_threshold(test_y, base_probs, alert_threshold)

metrics_rows = [
    ("ROC-AUC", round(test_roc, 4), 0.5000),
    ("PR-AUC", round(test_pr, 4), round(float(test_y.mean()), 4)),
    ("Log-Loss", round(test_logloss, 4), round(base_logloss, 4)),
    ("Brier Score", round(test_brier, 4), round(base_brier, 4)),
    ("Brier Skill Score (%)", round(brier_skill_pct, 2), 0.00),
    ("Śr. P(gol) dla y=1 (padł gol)", round(mean_p_pos, 4), round(base_prob, 4)),
    ("Śr. P(gol) dla y=0 (brak gola)", round(mean_p_neg, 4), round(base_prob, 4)),
    ("Separacja P(y=1) - P(y=0)", round(mean_p_pos - mean_p_neg, 4), 0.0000),
    ("Top 10% trafność goli", round(top10_rate, 4), round(float(test_y.mean()), 4)),
    ("Top 10% Lift", round(top10_lift, 2), 1.00),
    ("Top 20% trafność goli", round(top20_rate, 4), round(float(test_y.mean()), 4)),
    ("Top 20% Lift", round(top20_lift, 2), 1.00),
    (f"Precision @ próg {alert_threshold:.3f}", round(lstm_cls["precision"], 4), round(base_cls["precision"], 4)),
    (f"Recall (Czułość) @ próg {alert_threshold:.3f}", round(lstm_cls["recall"], 4), round(base_cls["recall"], 4)),
    (f"F1-score @ próg {alert_threshold:.3f}", round(lstm_cls["f1"], 4), round(base_cls["f1"], 4)),
    (f"Specificity @ próg {alert_threshold:.3f}", round(lstm_cls["specificity"], 4), round(base_cls["specificity"], 4)),
    (f"FPR (Fałszywe alarmy) @ próg {alert_threshold:.3f}", round(lstm_cls["fpr"], 4), round(base_cls["fpr"], 4)),
]
metrics_df = pd.DataFrame(metrics_rows, columns=["Metryka", "LSTM (20 cech)", "Baseline stały"])

print(f"Kalibracja Platta (na val): a = {platt_a:.3f}, b = {platt_b:+.3f}")
print(f"Próg alarmowy Top-{ALERT_TOP_PERCENT}% (z val): {alert_threshold:.2%} | Baza treningowa: {base_prob:.2%}")
print(
    f"Zbiór testowy: {bundle.test.meta['match_id'].nunique()} niewidzianych meczów, "
    f"{len(test_y):,} okien (gole: {int(test_y.sum())}, odsetek: {test_y.mean():.2%})"
)
print(
    f"Macierz pomyłek @ próg {alert_threshold:.2%}: "
    f"TP={lstm_cls['tp']} | FP={lstm_cls['fp']} | FN={lstm_cls['fn']} | TN={lstm_cls['tn']}"
)
display(metrics_df)

# Wykres diagnostyczny: (1) separacja rozkładów P(gol), (2) krzywa kalibracji w 10 koszykach.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
ax1.hist(test_probs[test_y == 0], bins=30, density=True, alpha=0.55, color="tab:gray", label="Brak gola (y=0)")
ax1.hist(test_probs[test_y == 1], bins=30, density=True, alpha=0.60, color="tab:orange", label=f"Gol w {bundle.horizon_minutes} min (y=1)")
ax1.axvline(base_prob, color="black", linestyle="--", linewidth=1.4, label=f"Baza ({base_prob:.1%})")
ax1.axvline(alert_threshold, color="tab:red", linestyle=":", linewidth=1.8, label=f"Próg alarmu ({alert_threshold:.1%})")
ax1.set_title("Rozkład przewidywanych P(gol) na zbiorze testowym")
ax1.set_xlabel("Skalibrowane P(gol)")
ax1.set_ylabel("Gęstość")
ax1.legend()
ax1.grid(alpha=0.25)

order_idx = np.argsort(test_probs)
bins = np.array_split(order_idx, 10)
bin_pred = [float(test_probs[b].mean()) for b in bins]
bin_obs = [float(test_y[b].mean()) for b in bins]
max_lim = max(max(bin_pred), max(bin_obs)) * 1.15
ax2.plot([0, max_lim], [0, max_lim], "k--", alpha=0.5, label="Idealna kalibracja (y=x)")
ax2.plot(bin_pred, bin_obs, marker="o", color="tab:blue", linewidth=2, label="LSTM (10 decyli)")
ax2.set_title("Krzywa kalibracji na zbiorze testowym")
ax2.set_xlabel("Średnie przewidywane P(gol)")
ax2.set_ylabel("Rzeczywista częstość goli")
ax2.legend()
ax2.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Krok 9 — Symulacja „ucinanych meczów na żywo” (Backtest na historycznych meczach testowych)

Ten krok odpowiada dokładnie scenariuszowi testowania na **niewidzianych w treningu meczach historycznych**, którym **ucinamy przyszłość** w konkretnej minucie `t` — model dostaje wyłącznie historię `0..t`, dokładnie tak jak przy meczu trwającym na żywo:
1. **Ucięcia `PRZED GOLEM (y=1)`**: wyznaczamy środek każdego rzeczywistego okna przed golem (ok. 4–5 minut przed bramką), gdzie z historii wiemy, że gol padł w `(t, t + M]`.
2. **Ucięcia `KONTROLNE (y=0)`**: z tych samych meczów losujemy tyle samo spokojnych momentów, po których przez `M` minut gol **nie padł** — żeby uczciwie zmierzyć zarówno trafienia (`Recall`), jak i fałszywe alarmy (`FPR`).

In [ ]:
test_view = bundle.test.meta.copy()
test_view["p_goal"] = test_probs
test_view["y_actual"] = test_y

match_pos_counts = test_view.groupby("match_id")["y_actual"].sum()
matches_with_goals = match_pos_counts[match_pos_counts > 0].sort_values(ascending=False)
replay_match_ids = [int(mid) for mid in matches_with_goals.index[: max(1, N_REPLAY_MATCHES)]]


def build_truncated_replay_cases(
    df_test: pd.DataFrame,
    match_ids: list[int],
    threshold: float,
    baseline_p: float,
    seed: int,
) -> pd.DataFrame:
    """Wybiera dla każdego meczu migawki PRZED GOLEM (y=1) oraz kontrolne KONTROLNA (y=0)."""
    records = []
    for mid in match_ids:
        m_df = df_test[df_test["match_id"] == mid].sort_values(["team_id", "period_value", "minute"]).copy()
        pos_snapshots = []
        for (_, _), grp in m_df.groupby(["team_id", "period_value"], sort=False):
            y_flag = (grp["y_actual"] >= 0.5).to_numpy()
            block_ids = (y_flag != np.roll(y_flag, 1))
            block_ids[0] = True
            grp = grp.assign(_block=np.cumsum(block_ids))
            for _, blk in grp[grp["y_actual"] >= 0.5].groupby("_block"):
                # Środek bloku to zwykle ~4-5 minut przed rzeczywistą bramką.
                pos_snapshots.append(blk.iloc[len(blk) // 2])

        neg_pool = m_df[m_df["y_actual"] < 0.5]
        n_ctrl = min(len(neg_pool), max(1, len(pos_snapshots)))
        ctrl_snapshots = []
        if n_ctrl > 0:
            rng = np.random.default_rng(seed + int(mid))
            chosen_idx = rng.choice(len(neg_pool), size=n_ctrl, replace=False)
            ctrl_snapshots = [neg_pool.iloc[int(i)] for i in chosen_idx]

        for label, rows_list in (("PRZED GOLEM (y=1)", pos_snapshots), ("KONTROLNA (y=0)", ctrl_snapshots)):
            for row in rows_list:
                p_val = float(row["p_goal"])
                records.append({
                    "Mecz": int(mid),
                    "Drużyna": str(row["team_name"]),
                    "Połowa": int(row["period_value"]),
                    "Ucięto w min. t": int(row["minute"]),
                    "Bilans": f"{int(round(row['score_diff'])):+d}",
                    "Typ migawki": label,
                    "P(gol) %": round(100.0 * p_val, 2),
                    "Lift vs baza": round(p_val / max(baseline_p, 1e-8), 2),
                    "Alarm?": "TAK" if p_val >= threshold else "nie",
                    "Gol w (t, t+M]?": int(round(row["y_actual"])),
                })

    if not records:
        return pd.DataFrame()
    return pd.DataFrame(records).sort_values(["Mecz", "Ucięto w min. t", "Drużyna"]).reset_index(drop=True)


replay_df = build_truncated_replay_cases(test_view, replay_match_ids, alert_threshold, base_prob, SEED)
if replay_df.empty:
    print("Brak meczów z golami w zbiorze testowym do symulacji.")
else:
    pos_cases = replay_df[replay_df["Gol w (t, t+M]?"] == 1]
    neg_cases = replay_df[replay_df["Gol w (t, t+M]?"] == 0]
    replay_auc = roc_auc_score_np(
        replay_df["Gol w (t, t+M]?"].to_numpy(dtype=np.float32),
        replay_df["P(gol) %"].to_numpy(dtype=np.float32),
    )
    print(f"=== Symulacja ucinanych meczów testowych ({len(replay_match_ids)} meczów niewidzianych w treningu) ===")
    print(f"  Ucięcia PRZED GOLEM (n={len(pos_cases)}): śr. P(gol)={pos_cases['P(gol) %'].mean():.2f}% | "
          f"powyżej bazy={int((pos_cases['Lift vs baza'] > 1.0).sum())}/{len(pos_cases)} | "
          f"alarm (Recall)={100.0 * (pos_cases['Alarm?'] == 'TAK').mean():.1f}%")
    if not neg_cases.empty:
        print(f"  Ucięcia KONTROLNE   (n={len(neg_cases)}): śr. P(gol)={neg_cases['P(gol) %'].mean():.2f}% | "
              f"fałszywe alarmy (FPR)={100.0 * (neg_cases['Alarm?'] == 'TAK').mean():.1f}%")
    print(f"  ROC-AUC na zbalansowanych migawkach z tych meczów: {replay_auc:.4f}")
    display(replay_df)

    # Zestawienie wszystkich minut dla wybranych meczów testowych.
    per_match_rows = []
    for mid in replay_match_ids:
        sub = test_view[test_view["match_id"] == mid]
        home_sub = sub[sub["is_home"] >= 0.5]
        title = f"{home_sub['team_name'].iloc[0]} vs {home_sub['opp_team_name'].iloc[0]}" if not home_sub.empty else str(mid)
        y_m = sub["y_actual"].to_numpy()
        p_m = sub["p_goal"].to_numpy()
        per_match_rows.append({
            "Mecz": mid,
            "Spotkanie": title,
            "Okna y=1": int((y_m >= 0.5).sum()),
            "Okna y=0": int((y_m < 0.5).sum()),
            "Śr. P(gol) y=1 (%)": round(100.0 * float(p_m[y_m >= 0.5].mean()), 2) if np.any(y_m >= 0.5) else np.nan,
            "Śr. P(gol) y=0 (%)": round(100.0 * float(p_m[y_m < 0.5].mean()), 2) if np.any(y_m < 0.5) else np.nan,
            "ROC-AUC w meczu": round(roc_auc_score_np(y_m, p_m), 4),
        })
    print("\nPodsumowanie całych wybranych meczów testowych (wszystkie minuty):")
    display(pd.DataFrame(per_match_rows))

## Krok 10 — Wizualizacja jednego meczu testowego

Wykres pokazuje przewidywane `P(gol w kolejnych M min)` minuta po minucie dla **gospodarzy**, **gości** oraz **którejkolwiek drużyny** na wybranym meczu ze zbioru testowego.

In [ ]:
test_view = bundle.test.meta.copy()
test_view["p_goal"] = test_probs
test_view["y_actual"] = test_y

selected_match_id = int(test_view.groupby("match_id")["y_actual"].sum().idxmax())
match_rows = test_view[test_view["match_id"] == selected_match_id].sort_values(["period_value", "minute"])

home_rows = match_rows[match_rows["is_home"] == 1.0].set_index("minute")
away_rows = match_rows[match_rows["is_home"] == 0.0].set_index("minute")
home_name = home_rows["team_name"].iloc[0]
away_name = away_rows["team_name"].iloc[0]

common_mins = home_rows.index.intersection(away_rows.index)
p_home = home_rows.loc[common_mins, "p_goal"]
p_away = away_rows.loc[common_mins, "p_goal"]
p_any = 1.0 - (1.0 - p_home) * (1.0 - p_away)

score_changes = home_rows["score_diff"].diff().fillna(0.0)
home_goal_mins = score_changes[score_changes > 0].index.tolist()
away_goal_mins = score_changes[score_changes < 0].index.tolist()

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(common_mins, 100 * p_home, label=f"{home_name} (Gospodarz)", color="tab:blue", linewidth=2)
ax.plot(common_mins, 100 * p_away, label=f"{away_name} (Gość)", color="tab:red", linewidth=2)
ax.plot(common_mins, 100 * p_any, label="Którakolwiek drużyna", color="black", linestyle="--", alpha=0.6)

for gm in home_goal_mins:
    ax.axvline(gm, color="tab:blue", linestyle=":", linewidth=1.8, alpha=0.8)
for gm in away_goal_mins:
    ax.axvline(gm, color="tab:red", linestyle=":", linewidth=1.8, alpha=0.8)

ax.set_title(f"Mecz {selected_match_id}: {home_name} vs {away_name} — P(gol w kolejnych {bundle.horizon_minutes} min)")
ax.set_xlabel("Rozszerzona minuta meczu (t)")
ax.set_ylabel("Przewidywane prawdopodobieństwo (%)")
ax.legend(loc="upper left")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Krok 11 — Zapis modelu (i opcjonalne pobranie na dysk)

Zapisujemy wagi modelu, statystyki skalera oraz parametry Platta do pliku `.pt`. Jeśli chcesz od razu pobrać plik z Colaba na swój komputer, zaznacz `DOWNLOAD_MODEL` w formularzu w Kroku 1.

In [ ]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "feature_names": list(bundle.feature_names),
    "feature_mean": bundle.feature_mean,
    "feature_std": bundle.feature_std,
    "sequence_length": bundle.sequence_length,
    "horizon_minutes": bundle.horizon_minutes,
    "hidden_size": HIDDEN_SIZE,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "platt_a": platt_a,
    "platt_b": platt_b,
}
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(checkpoint, MODEL_PATH)
print(f"Zapisano checkpoint modelu: {MODEL_PATH}")

if IN_COLAB and DRIVE_MOUNTED:
    drive_copy = Path(DRIVE_DATA_DIR) / MODEL_PATH.name
    if drive_copy.resolve() != MODEL_PATH.resolve():
        shutil.copy2(MODEL_PATH, drive_copy)
        print(f"Skopiowano również na Google Drive: {drive_copy}")

if IN_COLAB and DOWNLOAD_MODEL:
    from google.colab import files

    files.download(str(MODEL_PATH))